WITH / WITHOUT TEXTUAL REPRESENTATION

In [ ]:
import re
import json

# === JSON cleanup ===
def clean_answer(raw_response: str):
    """
    Cleans and safely parses LLM JSON output (even if wrapped in markdown fences).
    """
    # Remove Markdown code fences (```json ... ```)
    cleaned = re.sub(r"^```(?:json|python)?", "", raw_response.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r"```$", "", cleaned.strip())

    # Remove common markdown artifacts
    cleaned = cleaned.replace("\n", "\\n").replace("\r", "\\r")

    # Some models may prepend language tag or commentary
    cleaned = re.sub(r"^json", "", cleaned.strip(), flags=re.IGNORECASE)

    # Extra cleanup for accidental trailing commas
    cleaned = re.sub(r",\s*([\]\}])", r"\1", cleaned)

    # Try JSON parsing
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        # print(f"⚠️ JSON parsing failed, attempting manual recovery: {e}")
        # # Attempt a secondary cleaning
        cleaned2 = cleaned.replace('\\n', ' ').replace('\\"', '"')
        return json.loads(cleaned2)

WE NEED TO SAVE THE PHASE DATA AS A STRING WITHOUT APPLYING THE TEXTUAL REPRESENTATION PIPELINE

In [ ]:
# === Load dataset ===
with open("../../data/CDG-FCO/flight_data_with_minutes_since_start.json", "r") as f:
    flights_data = json.load(f)

In [ ]:
# # === Parameters ===

# all_results = []
# flight_index = {}

# # === Process flights in batches ===
# for flight in flights_data:
#     fr24_id = flight["flight_metadata"]["fr24_id"]
#     result_entry = {
#         "fr24_id": fr24_id,
#         "take_off": str(flight.get("take_off", [])),
#         "cruising": str(flight.get("cruising", [])),
#         "landing": str(flight.get("landing", []))
#     }
#     all_results.append(result_entry)

# # === Save output ===
# output_path = "flights_summary_without_textual_representation.json"
# with open(output_path, "w", encoding="utf-8") as f:
#     json.dump(all_results, f, indent=4, ensure_ascii=False)


In [ ]:
# !pip install openai faiss-cpu matplotlib

import os
import json
import faiss
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from openai import OpenAI

# --------------------------
# Config
# --------------------------
MODEL = "text-embedding-3-large"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY_2"))
K = 5  # number of most similar/dissimilar flights
PHASES = ["take_off", "cruising", "landing"]

# --------------------------
# Load summaries and data
# --------------------------
with open("flights_summary_without_textual_representation.json", "r", encoding="utf-8") as f:
    summaries = json.load(f)

with open("../../data/CDG-FCO/flight_data_separated_into_phases.json", "r", encoding="utf-8") as f:
    flights_data = json.load(f)

# Map fr24_id -> raw data per phase
data_by_id = {f["flight_metadata"]["fr24_id"]: f for f in flights_data}

# Map fr24_id -> summaries per phase, safely handling nulls
summary_by_phase = {}
for s in summaries:
    fid = s["fr24_id"]
    summary_by_phase[fid] = {}
    for phase in PHASES:
        text = s.get(phase)
        summary_by_phase[fid][phase] = text if isinstance(text, str) and text.strip() else None

common_ids = list(set(summary_by_phase.keys()) & set(data_by_id.keys()))
print(f"✅ Common flights: {len(common_ids)}")

# --------------------------
# Embeddings helper
# --------------------------
def get_embeddings(texts, model=MODEL, batch_size=64):
    vectors = []
    total = len(texts)
    for i in range(0, total, batch_size):
        batch = texts[i:i + batch_size]
        resp = client.embeddings.create(model=model, input=batch)
        batch_vecs = [np.array(e.embedding, dtype="float32") for e in resp.data]
        vectors.extend(batch_vecs)
    X = np.vstack(vectors)
    X = X / np.linalg.norm(X, axis=1, keepdims=True)  # L2 normalization
    return X

# --------------------------
# Build FAISS index per phase (robust to missing summaries)
# --------------------------
# phase_indices = {}
# phase_metas = {}

# for phase in PHASES:
#     print(f"\n🧠 Building FAISS index for phase: {phase}")
#     valid_flights = [fid for fid in common_ids if summary_by_phase[fid].get(phase)]
#     texts = [summary_by_phase[fid][phase] for fid in valid_flights]
#     if not texts:
#         print(f"⚠️ No valid flights for phase {phase}, skipping index.")
#         continue

#     E = get_embeddings(texts)
#     d = E.shape[1]
#     index = faiss.IndexFlatIP(d)
#     index.add(E)
#     phase_indices[phase] = index
#     phase_metas[phase] = valid_flights
#     print(f"✅ Phase '{phase}': added {len(valid_flights)} valid flights.")

# print("\n✅ All FAISS indices built successfully.\n")

In [ ]:
import os
import numpy as np
import json
import faiss
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY_2"))

# --------------------------
# CONFIGURATION
# --------------------------
EMBED_MODEL = "text-embedding-3-large"
LLM_MODEL = "gpt-4.1"
K = 5  # number of similar flights retrieved

# --------------------------
# LOAD DATA
# --------------------------
with open("../../data/CDG-FCO/flight_data_with_minutes_since_start.json", "r", encoding="utf-8") as f:
    flights_data = json.load(f)

with open("flights_summary_without_textual_representation.json", "r", encoding="utf-8") as f:
    summaries = json.load(f)

data_by_id = {f["flight_metadata"]["fr24_id"]: f for f in flights_data}
summary_by_phase = {
    s["fr24_id"]: {p: s.get(p, None) for p in ["take_off", "cruising", "landing"]}
    for s in summaries
}

print(f"✅ Loaded {len(flights_data)} flights with data and summaries.\n")


def save_phase_index(phase, index, ids, embeddings):
    faiss.write_index(index, f"faiss_index_{phase}.index")
    np.save(f"embeddings_{phase}.npy", embeddings)
    with open(f"ids_{phase}.json", "w") as f:
        json.dump(ids, f)
    print(f"💾 Saved FAISS index, embeddings and IDs for phase '{phase}'")


def load_phase_index(phase):
    index = faiss.read_index(f"faiss_index_{phase}.index")
    embeddings = np.load(f"embeddings_{phase}.npy")
    with open(f"ids_{phase}.json", "r") as f:
        ids = json.load(f)
    print(f"📂 Loaded FAISS index, embeddings and IDs for phase '{phase}'")
    return index, ids, embeddings

# --------------------------
# EMBEDDING HELPER
# --------------------------
def get_embeddings(texts, model=EMBED_MODEL, batch_size=32):
    """Embed multiple texts with normalization."""
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        resp = client.embeddings.create(model=model, input=batch)
        embs = [np.array(e.embedding, dtype="float32") for e in resp.data]
        vecs.extend(embs)
    X = np.vstack(vecs)
    X /= np.linalg.norm(X, axis=1, keepdims=True)
    return X

# --------------------------
# BUILD RAG INDEX FOR EACH PHASE
# --------------------------

def build_phase_index(phase, force_rebuild=False):
    # 1) If files exist → load them
    if (not force_rebuild and 
        os.path.exists(f"faiss_index_{phase}.index") and
        os.path.exists(f"embeddings_{phase}.npy") and
        os.path.exists(f"ids_{phase}.json")):

        return load_phase_index(phase)

    # 2) Else → rebuild the index
    valid = [fid for fid, s in summary_by_phase.items() if s.get(phase)]
    texts = [summary_by_phase[fid][phase] for fid in valid]

    if not texts:
        raise ValueError(f"No valid summaries for phase '{phase}'")

    E = get_embeddings(texts)
    d = E.shape[1]

    index = faiss.IndexFlatIP(d)
    index.add(E)

    print(f"🔧 Index built for phase '{phase}' with {len(valid)} flights")

    # 3) Save everything for next time
    save_phase_index(phase, index, valid, E)

    return index, valid, E

In [ ]:
phase_index_take_off, phase_ids_take_off, _ = build_phase_index("take_off")
phase_index_cruising, phase_ids_cruising, _ = build_phase_index("cruising")
phase_index_landing, phase_ids_landing, _ = build_phase_index("landing")

In [ ]:
index_dictionary = {"take_off": phase_index_take_off, "cruising": phase_index_cruising, "landing": phase_index_landing}
ids_dictionary = {"take_off": phase_ids_take_off, "cruising": phase_ids_cruising, "landing": phase_ids_landing}

In [ ]:
PREDICT_POSITION_PROMPT = f"""
A flight entered a GPS spoofing zone. Therefore his position is now unknown and your task is to help the pilot 
by predicting the flight's latitude and longitude.
Here is its known trajectory (before spoofing):

{{partial_summary}}

The following are summaries of {{number_similar_flight}} similar flights for the same phase:

{{sim_text}}

{{predictions_history}}

Based on these, predict the geographic position of the target flight {{predict_minutes}} minutes after the beginning of the phase.
Return your answer as JSON: {{{{"lat": "the predicted latitude", "lon": "the predicted longitude"}}}} only.
"""

# --------------------------
# PREDICT POSITION VIA LLM
# --------------------------
def predict_position_from_prompt(partial_summary, similar_summaries, predict_minutes, prediction_history: list = []):
    """Ask GPT to predict target position after N minutes using similar flights."""
    sim_text = "\n\n".join([f"Similar flight {i+1}:\n{txt}" for i, txt in enumerate(similar_summaries)])
    if not prediction_history:
        predictions_history = ""
    else:
        predictions_history = f"""Here are the predictions you already made. They are given as a list of triplet [latitude, longitude, number of minutes into the phase]. Use these past predictions to ensure that your next prediction is consistent.
Make sure the plane continues moving forward along its trajectory, does not remain at the same coordinates repeatedly, and does not move backward in a way that violates realistic flight dynamics.

PREDICTION HISTORY: {{prediction_history}}"""
        predictions_history = predictions_history.format(prediction_history=prediction_history)

    prompt = PREDICT_POSITION_PROMPT.format(
        partial_summary=partial_summary, 
        number_similar_flight=len(similar_summaries),
        sim_text=sim_text,
        predict_minutes=predict_minutes,
        predictions_history=predictions_history,
    )
        
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return clean_answer(response.choices[0].message.content), response.usage.total_tokens


In [ ]:
# --------------------------
# FIND SIMILAR FLIGHTS
# --------------------------
def find_similar_flights(partial_summary, target_id, phase, k=K):
    """Embed the target partial summary and find K most similar full summaries."""
    q_vec = get_embeddings([partial_summary])
    sims, idxs = index_dictionary[phase].search(q_vec, k)
    ids_for_phase = ids_dictionary[phase]
    sims, idxs = sims[0], idxs[0]
    neighbors = [(ids_for_phase[i], float(sims[i])) for i in range(len(idxs)) if ids_for_phase[i] != target_id]
    return neighbors


In [ ]:
import matplotlib.pyplot as plt
import math

def haversine(lat1, lon1, lat2, lon2):
    R = 6371e3
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

def predict_flight(flight_id, phase, with_feedback_loop=True, k=K):
    # --------------------------
    # 1️⃣ Extraire les données du vol cible
    # --------------------------
    target_flight = data_by_id[flight_id]
    phase_data = target_flight[phase]

    # --------------------------
    # 2️⃣ Définir la zone de spoofing
    # --------------------------
    N = len(phase_data)
    SPOOF_INDEX = N // 2   # floor division → produit exactement la séparation souhaitée

    # print(f"📏 Phase length = {N} points — Observed = 0:{SPOOF_INDEX-1}, Predicted = {SPOOF_INDEX}:{N-1}")

    observed_data = phase_data[:SPOOF_INDEX]
    spoofed_data  = phase_data[SPOOF_INDEX:]

    # --------------------------
    # 3️⃣ Générer le résumé partiel (RAG query)
    # --------------------------
    partial_summary = str(observed_data)
    # print("🧩 Résumé partiel généré.\n")

    # --------------------------
    # 5️⃣ Prédire tous les points de la zone spoofée
    # --------------------------
    predicted_positions = []
    ground_truth_positions = []

    for i, point in enumerate(spoofed_data):
        t_min = point["minutes_since_start"]
        barometric_altitude = point["altitude_ft"]

        # --------------------------
        # 4️⃣ Trouver les vols similaires en prenant en compte l'altitude connu
        # --------------------------
        altitude_prompt = f"""{t_min} minutes into the {phase} phase, the aircraft reached an altitude of {barometric_altitude} feet."""
        partial_summary = f"""{partial_summary} {altitude_prompt}"""
        neighbors = find_similar_flights(partial_summary, target_id=flight_id, phase=phase, k=k)
        similar_ids = [fid for fid, sim in neighbors]
        similar_summaries = [summary_by_phase[fid][phase] for fid in similar_ids]

        if with_feedback_loop:
            prediction, _ = predict_position_from_prompt(
                partial_summary,
                similar_summaries,
                predict_minutes=t_min,
                prediction_history=predicted_positions
            )
        else:
            prediction, _ = predict_position_from_prompt(
                partial_summary,
                similar_summaries,
                predict_minutes=t_min
            )

        if prediction:
            lat = round(float(prediction["lat"]), 2)
            lon = round(float(prediction["lon"]), 2)
            predicted_positions.append([lat, lon, t_min])
        ground_truth_positions.append([point["lat"], point["lon"], t_min])

    mean_error = 0

    for i in range(len(predicted_positions)):
        predicted_point = predicted_positions[i]
        lat1 = predicted_point[0]
        lon1 = predicted_point[1]
        ground_truth_point = ground_truth_positions[i]
        lat2 = ground_truth_point[0]
        lon2 = ground_truth_point[1]
        error = haversine(lat1, lon1, lat2, lon2)
        mean_error += error

    mean_error = mean_error / len(predicted_positions)

    return round(mean_error/1000, 1)


In [ ]:
# import json

# # Path to your JSON file
# input_file = "../../data/CDG-FCO/lstm_validation_flight_ids.json"

# with open(input_file, "r", encoding="utf-8") as f:
#     validation_flight_ids = json.load(f)

# print(f"Loaded {len(validation_flight_ids)} flight IDs from {input_file}")

# results = {"take_off": 0, "cruising": 0, "landing": 0}

In [ ]:
import random

def calculate_mean_haversine_error(flight_ids):
    errors_takeoff = []
    errors_cruising = []
    errors_landing = []

    counter_takeoff = 0
    counter_cruising = 0
    counter_landing = 0

    for flight_id in flight_ids["take_off"]:
        try:
            print(f"counter take_off {counter_takeoff}")
            error_takeoff = predict_flight(flight_id, "take_off", with_feedback_loop=True)
            errors_takeoff.append(error_takeoff)
            counter_takeoff += 1
            print(error_takeoff)
        except:
            print("ERROR RETRY")
            try:
                error_takeoff = predict_flight(flight_id, "take_off", with_feedback_loop=True)
                errors_takeoff.append(error_takeoff)
                counter_takeoff += 1
                print(error_takeoff)
            except:
                print("ERROR GO TO NEXT FLIGHT")
                continue

    for flight_id in flight_ids["cruising"]:
        try:
            print(f"counter cruising {counter_cruising}")
            error_cruising = predict_flight(flight_id, "cruising", with_feedback_loop=True)
            errors_cruising.append(error_cruising)
            counter_cruising += 1
            print(error_cruising)
        except:
            print("ERROR RETRY")
            try:
                error_cruising = predict_flight(flight_id, "cruising", with_feedback_loop=True)
                errors_cruising.append(error_cruising)
                counter_cruising += 1
                print(error_cruising)
            except:
                print("ERROR GO TO NEXT FLIGHT")
                continue

    for flight_id in flight_ids["landing"]:
        try:
            print(f"counter landing {counter_landing}")
            error_landing = predict_flight(flight_id, "landing", with_feedback_loop=True)
            errors_landing.append(error_landing)
            counter_landing += 1
            print(error_landing)
        except:
            print("ERROR RETRY")
            try:
                error_landing = predict_flight(flight_id, "landing", with_feedback_loop=True)
                errors_landing.append(error_landing)
                counter_landing += 1
                print(error_landing)
            except:
                print("ERROR GO TO NEXT FLIGHT")
                continue

    return errors_takeoff, errors_cruising, errors_landing




In [ ]:
import json

output_file = "HAVERSINE_ERROR_RAG_WITHOUT_TEXTUAL_REPRESENTATION.json"

with open("../../data/CDG-FCO/RESULTS/test_sample.json", "r", encoding="utf-8") as f:
    flight_ids_by_phase = json.load(f)

mean_error_takeoff, mean_error_cruising, mean_error_landing = calculate_mean_haversine_error(flight_ids_by_phase)

results = {
    "RAG": 
           {"take_off": mean_error_takeoff, 
            "cruising": mean_error_cruising, 
            "landing": mean_error_landing}
}

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print(f"✅ Saved to {output_file}")

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

# =========================
# Paths
# =========================
PATH_RAG = "../../data/CDG-FCO/RESULTS/MEAN_HAVERSINE_RAG.json"
PATH_RAG_NO_TEXT = "HAVERSINE_ERROR_RAG_WITHOUT_TEXTUAL_REPRESENTATION.json"

PHASES = ["take_off", "cruising", "landing"]

# =========================
# Load data
# =========================
with open(PATH_RAG, "r", encoding="utf-8") as f:
    rag_data = json.load(f)["RAG"]

with open(PATH_RAG_NO_TEXT, "r", encoding="utf-8") as f:
    rag_no_text = json.load(f)["RAG"]

# Extract only the error value (index 0) from [error, flight_id]

# =========================
# Prepare boxplot data
# =========================
box_data = []
positions = []

pos = 1
gap_between_phases = 1.6

for phase in PHASES:
    box_data.append(rag_data[phase])
    box_data.append(rag_no_text[phase])

    positions.extend([pos, pos + 0.6])
    pos += gap_between_phases + 0.6

# =========================
# Plot
# =========================
plt.figure(figsize=(11, 6))

bp = plt.boxplot(
    box_data,
    positions=positions,
    widths=0.5,
    patch_artist=True,
    showmeans=True
)

# Colors: same color per method across phases
colors = ["#4C72B0", "#DD8452"] * len(PHASES)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)

# Mean marker styling
for mean in bp["means"]:
    mean.set_marker("o")
    mean.set_markerfacecolor("black")
    mean.set_markeredgecolor("black")

# =========================
# X-axis (phase-centered)
# =========================
phase_centers = [
    np.mean(positions[i * 2:(i * 2) + 2])
    for i in range(len(PHASES))
]

plt.xticks(phase_centers, ["Take-off", "Cruising", "Landing"])
plt.ylabel("Haversine error (km)")
plt.title("Ablation Study – Effect of Removing Flight Textual Representation")

# Legend
plt.plot([], [], color="#4C72B0", label="RAG")
plt.plot([], [], color="#DD8452", label="RAG without textual representation")
plt.legend()

plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

import json
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

# =========================
# Paths
# =========================
OUTPUT_CSV = "RAG_RETRIEVAL_ABLATION_STATS.csv"

PHASES = ["take_off", "cruising", "landing"]

# =========================
# Load data
# =========================
with open(PATH_RAG, "r", encoding="utf-8") as f:
    rag_data = json.load(f)["RAG"]

with open(PATH_RAG_NO_TEXT, "r", encoding="utf-8") as f:
    rag_no_text = json.load(f)["RAG"]

# Extract only error values from [error, flight_id]
rag_data = {p: np.array(rag_data[p]) for p in PHASES}
rag_no_text_data = {
    p: np.array(rag_no_text[p])
    for p in PHASES
}

# =========================
# Helper: descriptive stats
# =========================
def describe(values):
    return {
        "mean": np.mean(values),
        "median": np.median(values),
        "std": np.std(values, ddof=1),
        "q1": np.percentile(values, 25),
        "q3": np.percentile(values, 75),
        "min": np.min(values),
        "max": np.max(values),
        "n": len(values),
    }

def round_stats(d, decimals=2):
    out = {}
    for k, v in d.items():
        if isinstance(v, (float, np.floating)):
            out[k] = round(v, decimals)
        else:
            out[k] = v
    return out

# =========================
# Collect stats
# =========================
rows = []

for phase in PHASES:
    for method, values in [
        ("RAG", rag_data[phase]),
        ("RAG_no_textual_representation", rag_no_text_data[phase]),
    ]:
        stats = describe(values)
        stats = round_stats(stats, decimals=2)

        rows.append({
            "phase": phase,
            "method": method,
            **stats
        })



# =========================
# Save CSV
# =========================
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)

print(f"✅ Statistical results written to:\n{OUTPUT_CSV}")
